In [ ]:
import datetime
import logging  # 追加：ログ制御用
import warnings  # 追加：警告非表示用
import pandas as pd
import yfinance as yf
from IPython.display import display, HTML

# 警告メッセージ（FutureWarningなど）を非表示にする
warnings.filterwarnings("ignore")

# yfinanceの内部エラー出力を非表示にする
yf_logger = logging.getLogger('yfinance')
yf_logger.setLevel(logging.CRITICAL)

# Define the lookback periods (in business days) for performance calculation.
LOOKBACK_DAYS = {
    "1 Day (%)": -2,
    "3 Days (%)": -4,
    "1 Week (%)": -6,
    "1 Month (%)": -21,
    "6 Months (%)": -126,
    "1 Year (%)": -252,
}

# =====================================================================
# DATA PROCESSING FUNCTION (Supports Market Segmentation)
# =====================================================================
def show_market_performances(market_dict):
    today = datetime.date.today()
    start_date = today - datetime.timedelta(days=60)
    
    # Format numerical values to 2 decimal places
    format_config = {"Latest Price": "{:,.2f}"}
    for column_name in LOOKBACK_DAYS.keys():
        format_config[column_name] = "{:+.2f}%"
        
    # ハイライト用の条件関数
    def highlight_extreme_values(val):
        if pd.isna(val):
            return ''
        if val >= 10:
            return 'background-color: #ffcccc; color: #cc0000; font-weight: bold;'
        elif val <= -10:
            return 'background-color: #cce5ff; color: #004085; font-weight: bold;'
        return ''
        
    # Process each market section
    for market_name, ticker_dict in market_dict.items():
        records = []
        
        # Display market header
        display(HTML(f"<h3 style='margin-top: 20px; color: #333;'>■ {market_name} Market</h3>"))
        
        for name, ticker in ticker_dict.items():
            try:
                # 【修正】 ignore_tz=True を追加してタイムゾーン警告を抑制
                df = yf.download(ticker, start=start_date, end=today, progress=False, ignore_tz=True)
                
                if df.empty:
                    # 元のコードにあったWarning表示も邪魔ならコメントアウトしてください
                    # print(f"[Warning] No data found for: {name} ({ticker})")
                    continue
                    
                prices = df["Close"].squeeze()
                latest_price = prices.iloc[-1]
                
                row = {
                    "Name": name,
                    "Ticker": ticker,
                    "Latest Price": latest_price
                }
                
                for label, index in LOOKBACK_DAYS.items():
                    if len(prices) >= abs(index):
                        past_price = prices.iloc[index]
                    else:
                        past_price = prices.iloc[0]
                    
                    row[label] = ((latest_price - past_price) / past_price) * 100
                    
                records.append(row)
                
            except Exception as e:
                # エラー時のprintも非表示にしたい場合はコメントアウトしてください
                pass
        
        if records:
            df_performance = pd.DataFrame(records)
            
            # .map を使用
            styled_df = (df_performance.style
                         .format(format_config)
                         .map(highlight_extreme_values, subset=list(LOOKBACK_DAYS.keys()))
                         .hide(axis="index"))
            
            display(styled_df)
        else:
            print("No data available for this market.")

In [ ]:
# =====================================================================
# CONFIGURATION & EXECUTION (Segmented by Market)
# =====================================================================
MARKET_TICKERS = {
    "United States": {
        "S&P 500": "^GSPC",
        "NASDAQ Composite": "^IXIC",
        "Apple": "AAPL",
        "NVIDIA": "NVDA",
        "Microsoft": "MSFT",
    },
    "Japan": {
        "Nikkei 225": "^N225",
        "Toyota Motor": "7203.T",
        "Sony Group": "6758.T",
        "Mitsubishi UFJ Financial": "8306.T",
    },
    "Singapore": {
        "STI Index": "^STI",
        "DBS Group": "D05.SI",
        "UOB": "U11.SI",
        "Singtel": "Z74.SI",
        "CapitaLand Ascendas REIT": "A17U.SI",
    },
    "Japan Stock": {
        "小松製作所 (6301)": "6301.T",
        "ニデック (6594)": "6594.T",
        "未来工業 (7931)": "7931.T",
        "東部ネットワーク (9036)": "9036.T",
        "ニトリHD (9843)": "9843.T",
        "MRK HD (9980)": "9980.T",
    },
    "US Stock (ETF)": {
        "iシェアーズ コア米国総合債券ETF (AGG)": "AGG"
    }
}

# Run and display the segmented tables
show_market_performances(MARKET_TICKERS)

In [ ]:
MARKET_TICKERS = {
    "Singapore Healthcare": {
        # --- 病院経営・総合クリニック ---
        "Raffles Medical Group (ラッフルズ医療)": "BSL.SI",  # 【修正】R01.SI から変更
        "IHH Healthcare (マウント・エリザベス等経営)": "Q0F.SI",  # 【修正】I01.SI から変更
        "Thomson Medical Group (トムソン医療)": "A50.SI",
        
        # --- 医療不動産（REIT / 病院・介護施設） ---
        "Parkway Life REIT (パークウェイ・ライフ)": "C2PU.SI",
        "First REIT (ファースト・リート)": "AW9U.SI",
        
        # --- 専門医療・歯科クリニックチェーン ---
        "Q&M Dental Group (歯科大手チェーン)": "QC7.SI",
        "Alliance Healthcare Group (総合診療・クリニック)": "MIJ.SI",  # 【修正】1D8から変更
        "Medinex Limited (医療サポート・クリニック運営)": "OTX.SI",
        
        # --- 医療用消耗品・製薬 ---
        "Medtecs International (医療用消耗品)": "546.SI",
        "Hyphens Pharma (医薬品・皮膚科製品)": "1J5.SI",
        "Riverstone": "AP4.SI"
    }
}
show_market_performances(MARKET_TICKERS)

In [ ]:
MARKET_TICKERS = {
"Singapore Food & Agribusiness": {
        "Wilmar International (ウィルマー・世界最大級のパーム油)": "F34.SI",
        "Olam Group (オラム・ココア、コーヒー、ナッツ等大手)": "VC2.SI",
        "Riverstone": "AP4.SI",
        "ヤンジジャン・シップビルディング(BS6)": "BS6.SI",
        "Sembcorp Ind (U96)": "U96.SI",
    }
}
show_market_performances(MARKET_TICKERS)